## Match Descriptors Between Two Views

# Feature Matching and Descriptor Alignment

Welcome to Unit 4 of our course. Up to this point, you have learned how to preprocess images and extract visual fingerprints by detecting keypoints and computing their descriptors.

Now, we are ready to tackle the first real dependency in our image stitching pipeline: **feature matching**. Finding keypoints in an isolated image is a great start, but to stitch a panorama, we must find the exact same points across two different views. By matching these descriptors, we create the critical links needed to eventually align and stitch images.

In this lesson, we will build a reliable feature matcher using OpenCV. We will configure our matcher based on descriptor data type and use Lowe's Ratio Test, a powerful technique for filtering out ambiguous matches.

---

## Handling Different Descriptor Types

Different algorithms produce different descriptor formats. SIFT produces arrays of floating-point numbers. ORB and AKAZE produce compact binary descriptors, represented as 8-bit unsigned integers.

Because our pipeline supports all three methods, our matcher needs to inspect the descriptor type before choosing a matching strategy.

```python
import numpy as np

def descriptors_are_binary(descriptors):
    return descriptors is not None and descriptors.dtype == np.uint8

```

If the descriptor `dtype` is `np.uint8`, we treat it as binary. Otherwise, we treat it as a floating-point descriptor such as SIFT.

---

## Lowe's Ratio Test

When matching descriptors, the algorithm computes distances between numeric fingerprints. A shorter distance means two descriptors look more similar.

For each descriptor in the first image, we ask for the two nearest candidates in the second image. The best candidate should be clearly better than the runner-up. If the two candidates are too close in quality, the match is ambiguous and should be rejected.

**Clear Match Example:**

```text
Descriptor from image A
       |
       |-- best candidate in image B      distance = 30
       |-- second-best candidate in B     distance = 80

Ratio check: best distance < ratio * second-best distance
With ratio = 0.75:
30 < 0.75 * 80
30 < 60  -> keep the match

```

**Ambiguous Match Example:**

```text
Descriptor from image A
       |
       |-- best candidate in image B      distance = 55
       |-- second-best candidate in B     distance = 60

With ratio = 0.75:
55 < 0.75 * 60
55 < 45  -> reject the match

```

The ratio acts like a strictness dial:

* **Lower values** (such as `0.65`) are stricter and produce fewer matches.
* **Higher values** (such as `0.85`) are looser and may keep more false matches.

The core filtering loop is short:

```python
good = []
for pair in raw_matches:
    if len(pair) == 2:
        best, second = pair
        if best.distance < ratio * second.distance:
            good.append(best)

```

---

## Understanding the FLANN Matcher

When we match descriptors, we are searching for the nearest neighbor in a high-dimensional space. For example, a SIFT descriptor is a vector of 128 numbers. If we have 2,000 keypoints in each image, a Brute-Force matcher would perform 4,000,000 comparisons ($2{,}000 \times 2{,}000$), which is computationally expensive.

To solve this, we use the **Fast Library for Approximate Nearest Neighbors (FLANN)**.

FLANN is a library of algorithms optimized for fast search in large datasets. Instead of checking every single possibility, it uses clever data structures — like trees or hash tables — to quickly narrow down the best candidates.

The "Approximate" in its name is key: it might not always find the absolute closest neighbor, but it finds a "good enough" neighbor significantly faster than a brute-force approach. In the context of image stitching, where we process thousands of points, this trade-off between perfect accuracy and high speed is essential for a responsive pipeline.

---

## Implementing the Matcher and Visualizing

Now, let's build our full `match_descriptors` function step by step. First, we will check our inputs and use our helper function to see if we have binary descriptors.

```python
import cv2
import numpy as np

def match_descriptors(des1, des2, ratio=0.75):
    if des1 is None or des2 is None:
        return []
    if len(des1) < 2 or len(des2) < 2:
        return []

    binary = descriptors_are_binary(des1) or descriptors_are_binary(des2)

```

Next, we need to configure OpenCV's Fast Library for Approximate Nearest Neighbors (FLANN) matcher. This is a highly optimized matcher, but it requires specific configuration dictionaries based on our data type.

```python
    if binary:
        # Settings for binary descriptors (ORB, AKAZE)
        index_params = dict(
            algorithm=6,
            table_number=6,
            key_size=12,
            multi_probe_level=1,
        )
        des1 = np.asarray(des1, dtype=np.uint8)
        des2 = np.asarray(des2, dtype=np.uint8)
    else:
        # Settings for floating-point descriptors (SIFT)
        index_params = dict(algorithm=1, trees=5)
        des1 = np.asarray(des1, dtype=np.float32)
        des2 = np.asarray(des2, dtype=np.float32)

    matcher = cv2.FlannBasedMatcher(index_params, dict(checks=50))

```

Here, we provide the exact algorithm parameters that OpenCV requires to process either binary or floating-point data efficiently. We also ensure our arrays are converted to the correct `dtype` format.

Now, we can ask the matcher for our top two candidates, apply the ratio test loop we learned earlier, and sort the results.

```python
    raw_matches = matcher.knnMatch(des1, des2, k=2)
    good = []
    for pair in raw_matches:
        if len(pair) == 2:
            best, second = pair
            if best.distance < ratio * second.distance:
                good.append(best)

    # Sort the matches so the best ones (shortest distance) are first
    return sorted(good, key=lambda match: match.distance)

```

By returning the matches sorted by `match.distance`, the most confident pairs are always placed at the beginning of our list.

To verify our work, we can use OpenCV's `cv2.drawMatches` function. This takes our two images, their keypoints, and our list of good matches, and draws lines connecting the corresponding features.

```python
# Assuming left and right images, along with their keypoints (kp) and descriptors (des) are ready
matches = match_descriptors(des1, des2, ratio=0.75)

print("method: sift")
print("ratio: 0.75")
print("left keypoints:", len(kp1))
print("right keypoints:", len(kp2))
print("good matches:", len(matches))

preview = cv2.drawMatches(
    left, kp1,
    right, kp2,
    matches[:60], # Draw only the top 60 to avoid clutter
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

```

When you run this on a pair of images, your output will look something like this:

```text
method: sift
ratio: 0.75
left keypoints: 2000
right keypoints: 2000
good matches: 438

```

The resulting preview image will beautifully display the two photos side by side with colorful lines connecting the exact same locations in both views.

---

## Summary and Next Steps

In this lesson, you took a major step forward in building your image stitcher. You learned how to inspect descriptor data types, configure a FLANN matcher for floating-point or binary descriptors, and apply Lowe's Ratio Test to filter unreliable matches.

This matching process provides the essential links we need. In our next and final unit, we will turn matching into a repeatable diagnostic report so we can warn the user before attempting more fragile geometry steps such as homography estimation.

Now, head over to the upcoming practice exercises. You will write the matching logic yourself and experiment with how different ratio values affect the number and quality of good matches.

## Choosing the Right Feature Detector

Welcome to the start of your matching report journey! Before two images can be matched, you need descriptors from both images. This first exercise quickly rebuilds the detector factory from the previous unit so that the matching code has reliable features to work with.

Open features.py and complete the create_detector function so that it returns a detector based on the method argument.

Here is what to handle, in order:

    For "sift", check whether cv2 has the SIFT_create attribute. If it does not, raise a ValueError with a clear message; otherwise, return cv2.SIFT_create(nfeatures=nfeatures).
    For "orb", return cv2.ORB_create(nfeatures=nfeatures).
    For "akaze", return cv2.AKAZE_create() (no nfeatures argument here).
    For anything else, raise a ValueError whose message includes the unknown method name.

This is a short review step, but it belongs in the matching workflow because matching cannot happen until both images have descriptors.

```
import cv2


def create_detector(method="sift", nfeatures=2000):
    # TODO: If method is "sift", first check that SIFT is available
    # in this OpenCV build using hasattr(cv2, "SIFT_create").
    # If it is not available, raise a ValueError with a clear message.
    # Otherwise, return cv2.SIFT_create(nfeatures=nfeatures).

    # TODO: If method is "orb", return cv2.ORB_create(nfeatures=nfeatures).

    # TODO: If method is "akaze", return cv2.AKAZE_create().
    # Note: AKAZE does NOT accept the nfeatures argument.

    # TODO: If none of the above matched, raise a ValueError that
    # includes the unknown method name in the message.
    pass


if __name__ == "__main__":
    # Quick check: run this file directly to see if your detector is created.
    detector = create_detector()
    print("Created detector:", type(detector).__name__)

```

Here is the completed `create_detector` function in `features.py`:

```python
import cv2


def create_detector(method="sift", nfeatures=2000):
    method = str(method).lower()

    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)

    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)

    if method == "akaze":
        return cv2.AKAZE_create()

    raise ValueError(f"Unknown feature detection method: {method}")


if __name__ == "__main__":
    # Quick check: run this file directly to see if your detector is created.
    detector = create_detector()
    print("Created detector:", type(detector).__name__)

```

## Detecting Keypoints and Computing Descriptors

Nice work setting up the detector factory in the last step. Now, it is time to put it to use and extract keypoints and descriptors from an image.

Inside features.py, fill in the body of detect_and_compute by following the TODO comments:

    Build a detector using create_detector, forwarding both the method and nfeatures arguments.
    Call the detector's detectAndCompute method on the grayscale image, passing None as the mask.
    Return the keypoints and descriptors together.

This is another short review step from the previous unit, but it is necessary here: the matcher in the next exercise receives descriptor arrays, not raw images. Once this function is working, the rest of the matching pipeline can begin to come together.

```
import cv2


def create_detector(method="sift", nfeatures=2000):
    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)

    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)

    if method == "akaze":
        return cv2.AKAZE_create()

    raise ValueError(f"Unknown feature method: {method}")


def detect_and_compute(gray, method="sift", nfeatures=2000):
    # TODO: Create a detector by calling create_detector with the given
    # method and nfeatures arguments.

    # TODO: Call the detector's detectAndCompute method on the grayscale
    # image. Pass None as the mask argument. It returns two values:
    # keypoints and descriptors.

    # TODO: Return the keypoints and descriptors as a tuple.
    pass


if __name__ == "__main__":
    # Quick check: run this file directly to confirm the detector factory
    # still works. detect_and_compute needs a real image to run, so it
    # is best tested through the unit tests.
    detector = create_detector()
    print("Created detector:", type(detector).__name__)

```

Here is the fixed code. The extra `hasattr`/`cv2.AKAZE.create()` checks have been removed so it strictly calls `cv2.AKAZE_create()` as required by the exercise environment:

```python
import cv2


def create_detector(method="sift", nfeatures=2000):
    method = str(method).lower()

    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)

    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)

    if method == "akaze":
        return cv2.AKAZE_create()

    raise ValueError(f"Unknown feature method: {method}")


def detect_and_compute(gray, method="sift", nfeatures=2000):
    detector = create_detector(method=method, nfeatures=nfeatures)
    keypoints, descriptors = detector.detectAndCompute(gray, None)
    return keypoints, descriptors


if __name__ == "__main__":
    detector = create_detector()
    print("Created detector:", type(detector).__name__)

```

## Matching Descriptors with Lowes Ratio Test

Nice work wiring up the detector and extracting real descriptors in the last step. Now comes the fun part — pairing those descriptors across two images to determine which points belong together.

In features.py, two functions are waiting for you: descriptors_are_binary and match_descriptors. The first is a small helper that determines whether a descriptor array uses the np.uint8 dtype (the format that binary detectors like ORB and AKAZE produce).

The second function is where the actual matching occurs. Follow the TODO comments within it to:

    Guard against None inputs and arrays that are too small for k=2 matching.
    Select the correct FLANN index_params depending on whether the descriptors are binary or floating-point, and cast the arrays to the matching dtype.
    Use algorithm=6 for LSH when matching binary descriptors, with table_number=6, key_size=12, and multi_probe_level=1.
    Use algorithm=1 with trees=5 for KD-tree matching when descriptors are floating-point.
    Build a cv2.FlannBasedMatcher, run knnMatch with k=2, and apply Lowe's ratio test to keep only confident matches.
    Return the surviving matches sorted by distance so that the strongest ones come first.

Once this is in place, you'll have a matching engine that works for both SIFT and ORB descriptors — a major step toward the final panorama pipeline.

```
import cv2
import numpy as np


def create_detector(method="sift", nfeatures=2000):
    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)

    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)

    if method == "akaze":
        return cv2.AKAZE_create()

    raise ValueError(f"Unknown feature method: {method}")


def detect_and_compute(gray, method="sift", nfeatures=2000):
    detector = create_detector(method=method, nfeatures=nfeatures)
    keypoints, descriptors = detector.detectAndCompute(gray, None)
    return keypoints, descriptors


def descriptors_are_binary(descriptors):
    # TODO: Return True only when descriptors is not None AND its dtype
    # is np.uint8 (the dtype used by binary descriptors like ORB/AKAZE).
    # Return False otherwise.
    pass


def match_descriptors(des1, des2, ratio=0.75):
    # TODO: If either des1 or des2 is None, return an empty list.

    # TODO: If either descriptor array has fewer than 2 entries, return
    # an empty list. FLANN's knnMatch with k=2 needs at least 2 items.

    # TODO: Use descriptors_are_binary to check if either array is
    # binary. Store the result in a variable named `binary`.

    # TODO: If binary is True, build index_params for binary descriptors
    # using algorithm=6 (LSH) with table_number=6, key_size=12, and
    # multi_probe_level=1. Then cast both des1 and des2 to np.uint8.
    # Otherwise, build index_params for floating-point descriptors using
    # algorithm=1 (KD-tree) with trees=5, and cast both des1 and des2
    # to np.float32.

    # TODO: Create a cv2.FlannBasedMatcher using your index_params and
    # dict(checks=50) as the search params.

    # TODO: Call matcher.knnMatch(des1, des2, k=2) to get the top two
    # candidates for each descriptor in des1.

    # TODO: Loop through the raw matches. For each pair, make sure it
    # contains exactly 2 elements (best, second). Apply Lowe's ratio
    # test: keep the best match only when
    #     best.distance < ratio * second.distance

    # TODO: Return the kept matches sorted by match.distance (ascending),
    # so the most confident matches come first.
    pass


if __name__ == "__main__":
    # Quick check: run this file directly to confirm the detector factory
    # still works. The matching functions need real descriptors to run,
    # so they are best tested through the unit tests.
    detector = create_detector()
    print("Created detector:", type(detector).__name__)

```

Here is the complete implementation for `descriptors_are_binary` and `match_descriptors` in `features.py`:

```python
import cv2
import numpy as np


def create_detector(method="sift", nfeatures=2000):
    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)

    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)

    if method == "akaze":
        return cv2.AKAZE_create()

    raise ValueError(f"Unknown feature method: {method}")


def detect_and_compute(gray, method="sift", nfeatures=2000):
    detector = create_detector(method=method, nfeatures=nfeatures)
    keypoints, descriptors = detector.detectAndCompute(gray, None)
    return keypoints, descriptors


def descriptors_are_binary(descriptors):
    return descriptors is not None and descriptors.dtype == np.uint8


def match_descriptors(des1, des2, ratio=0.75):
    # Guard against None inputs
    if des1 is None or des2 is None:
        return []

    # Guard against arrays with fewer than 2 entries required for k=2 matching
    if len(des1) < 2 or len(des2) < 2:
        return []

    # Check if either descriptor array is binary
    binary = descriptors_are_binary(des1) or descriptors_are_binary(des2)

    # Configure FLANN index parameters and cast dtypes based on descriptor format
    if binary:
        index_params = dict(
            algorithm=6,
            table_number=6,
            key_size=12,
            multi_probe_level=1,
        )
        des1 = np.asarray(des1, dtype=np.uint8)
        des2 = np.asarray(des2, dtype=np.uint8)
    else:
        index_params = dict(
            algorithm=1,
            trees=5,
        )
        des1 = np.asarray(des1, dtype=np.float32)
        des2 = np.asarray(des2, dtype=np.float32)

    # Initialize FLANN matcher
    matcher = cv2.FlannBasedMatcher(index_params, dict(checks=50))

    # Retrieve top 2 nearest candidates
    raw_matches = matcher.knnMatch(des1, des2, k=2)

    # Filter candidates using Lowe's ratio test
    good = []
    for pair in raw_matches:
        if len(pair) == 2:
            best, second = pair
            if best.distance < ratio * second.distance:
                good.append(best)

    # Return surviving matches sorted ascending by distance
    return sorted(good, key=lambda match: match.distance)


if __name__ == "__main__":
    detector = create_detector()
    print("Created detector:", type(detector).__name__)

```

## Wiring Up the Matching Pipeline

With the matching engine ready inside features.py, it is time to put it to work in a small command-line tool that produces a real matching report.

Open solution.py and fill in the main function step by step, following the TODO comments. The script should accept two image paths and a couple of options, run the full pipeline, and show the matched pairs side by side.

Here is the plan:

    Parse the command-line arguments with argparse: left, right, --method (choices: sift, orb, akaze, default sift), and --ratio (float, default 0.75).
    Read both images with read_color, then convert them to grayscale with preprocess_for_features.
    Run detect_and_compute on each grayscale image to get kp1, des1 and kp2, des2.
    Call match_descriptors with the chosen ratio and store the result in matches.
    Print the report lines in this exact order: method:, ratio:, left keypoints:, right keypoints:, good matches:.
    Build a preview with cv2.drawMatches using only the top 60 matches and the flag cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS.
    Show the preview with cv2.imshow, wait for a key with cv2.waitKey(0), and clean up with cv2.destroyAllWindows().

Once this script is complete, run it on an image pair and check the terminal output first. Confirm that the method and ratio match your command-line flags, then compare the left and right keypoint counts against the number of good matches. After that, inspect the preview lines to see whether the matches look geometrically plausible.


```
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors


def main():
    # TODO: Parse command-line arguments using argparse. You need:
    #   - "left" (positional, path to the left image)
    #   - "right" (positional, path to the right image)
    #   - "--method" with choices ["sift", "orb", "akaze"], default "sift"
    #   - "--ratio" as a float, default 0.75
    # Store the parsed values in a variable named `args`.

    # TODO: Read both images from disk using read_color and the paths
    # in args.left and args.right.

    # TODO: Preprocess both images into grayscale arrays using
    # preprocess_for_features.

    # TODO: Detect keypoints and descriptors for both images using
    # detect_and_compute with method=args.method.
    # Save the results as kp1, des1 (left) and kp2, des2 (right).

    # TODO: Match the descriptors using match_descriptors with
    # ratio=args.ratio. Store the result in a variable named `matches`.

    # TODO: Print a small report with the following lines (in this order):
    #   method: <method>
    #   ratio: <ratio>
    #   left keypoints: <number of left keypoints>
    #   right keypoints: <number of right keypoints>
    #   good matches: <number of matches>

    # TODO: Build a preview image with cv2.drawMatches. Use:
    #   - left, kp1, right, kp2 as the inputs
    #   - matches[:60] so you only draw the top 60 matches
    #   - None as the output image
    #   - flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS

    # TODO: Display the preview with cv2.imshow (use any window name),
    # then call cv2.waitKey(0) to wait for a key press, and finally
    # cv2.destroyAllWindows() to clean up.
    pass


if __name__ == "__main__":
    main()

```

Here is the completed `solution.py` implementation:

```python
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors


def main():
    # 1. Parse command-line arguments
    parser = argparse.ArgumentParser()
    parser.add_argument("left", help="Path to the left image")
    parser.add_argument("right", help="Path to the right image")
    parser.add_argument(
        "--method",
        choices=["sift", "orb", "akaze"],
        default="sift",
        help="Feature detection method (default: sift)",
    )
    parser.add_argument(
        "--ratio",
        type=float,
        default=0.75,
        help="Lowe's ratio test threshold (default: 0.75)",
    )
    args = parser.parse_args()

    # 2. Read both images from disk
    left = read_color(args.left)
    right = read_color(args.right)

    # 3. Preprocess both images into grayscale
    left_gray = preprocess_for_features(left)
    right_gray = preprocess_for_features(right)

    # 4. Detect keypoints and descriptors
    kp1, des1 = detect_and_compute(left_gray, method=args.method)
    kp2, des2 = detect_and_compute(right_gray, method=args.method)

    # 5. Match descriptors using Lowe's ratio test
    matches = match_descriptors(des1, des2, ratio=args.ratio)

    # 6. Print matching report
    print(f"method: {args.method}")
    print(f"ratio: {args.ratio}")
    print(f"left keypoints: {len(kp1)}")
    print(f"right keypoints: {len(kp2)}")
    print(f"good matches: {len(matches)}")

    # 7. Draw matches preview (top 60 matches)
    preview = cv2.drawMatches(
        left,
        kp1,
        right,
        kp2,
        matches[:60],
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )

    # 8. Display result window and wait for keypress
    cv2.imshow("matches preview", preview)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

## Tuning the Matching Report Defaults

Nice job wiring up the full matching pipeline in the previous exercise! Now, it's time to put on your tuning hat and tweak a few default values in features.py to see how they shape the matching report.

The functions themselves are already complete, so you only need to update three default arguments. Look for the TODO comments and follow them:

    In create_detector, increase the default nfeatures from 2000 to 4000.
    In detect_and_compute, increase the default nfeatures from 2000 to 4000 as well, so the new value reaches create_detector.
    In match_descriptors, decrease the default ratio from 0.75 to a stricter 0.65.

Once you save your changes, run solution.py with two image paths and check the terminal report. Compare left keypoints, right keypoints, and good matches before and after the default changes. Then inspect the preview image: more detected keypoints can give the matcher more opportunities, while a stricter ratio can reduce questionable matches. This small tuning step closes out the matching report tool and prepares you for the stitching work ahead.


```
import cv2
import numpy as np

# Note: The rest of this file is already complete from the previous
# exercises. For this final task, you will only adjust a few default
# values and then run solution.py to see how the matching report changes.


# TODO: Change the default value of `nfeatures` from 2000 to 4000 so the
# detector is allowed to find more keypoints per image.
def create_detector(method="sift", nfeatures=2000):
    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)

    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)

    if method == "akaze":
        return cv2.AKAZE_create()

    raise ValueError(f"Unknown feature method: {method}")


# TODO: Change the default value of `nfeatures` from 2000 to 4000 here as
# well, so the new value reaches create_detector when solution.py calls
# detect_and_compute without passing nfeatures explicitly.
def detect_and_compute(gray, method="sift", nfeatures=2000):
    detector = create_detector(method=method, nfeatures=nfeatures)
    keypoints, descriptors = detector.detectAndCompute(gray, None)
    return keypoints, descriptors


def descriptors_are_binary(descriptors):
    return descriptors is not None and descriptors.dtype == np.uint8


# TODO: Change the default value of `ratio` from 0.75 to a stricter 0.65
# so Lowe's ratio test keeps only highly confident matches.
def match_descriptors(des1, des2, ratio=0.75):
    if des1 is None or des2 is None:
        return []

    if len(des1) < 2 or len(des2) < 2:
        return []

    binary = descriptors_are_binary(des1) or descriptors_are_binary(des2)

    if binary:
        index_params = dict(
            algorithm=6,
            table_number=6,
            key_size=12,
            multi_probe_level=1,
        )
        des1 = np.asarray(des1, dtype=np.uint8)
        des2 = np.asarray(des2, dtype=np.uint8)
    else:
        index_params = dict(algorithm=1, trees=5)
        des1 = np.asarray(des1, dtype=np.float32)
        des2 = np.asarray(des2, dtype=np.float32)

    matcher = cv2.FlannBasedMatcher(index_params, dict(checks=50))
    raw_matches = matcher.knnMatch(des1, des2, k=2)

    good = []
    for pair in raw_matches:
        if len(pair) == 2:
            best, second = pair
            if best.distance < ratio * second.distance:
                good.append(best)

    return sorted(good, key=lambda match: match.distance)

# This file is already complete. It is the command-line tool you built
# in the previous exercise and you do not need to change anything here.
# Run it from the terminal with two image paths to see how your edits
# in features.py affect the matching report and the preview image.
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    # This default is synced with the intended final value in features.py
    parser.add_argument("--ratio", type=float, default=0.65)
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    left_gray = preprocess_for_features(left)
    right_gray = preprocess_for_features(right)

    kp1, des1 = detect_and_compute(left_gray, method=args.method)
    kp2, des2 = detect_and_compute(right_gray, method=args.method)
    matches = match_descriptors(des1, des2, ratio=args.ratio)

    print("method:", args.method)
    print("ratio:", args.ratio)
    print("left keypoints:", len(kp1))
    print("right keypoints:", len(kp2))
    print("good matches:", len(matches))

    preview = cv2.drawMatches(
        left,
        kp1,
        right,
        kp2,
        matches[:60],
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )

    cv2.imshow("matches", preview)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```